<h1>Bronze Bus Live Feed<h1>

<h2>Imports<h2>

<h3>Spark and Initialization<h3>

In [8]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Warsaw_Bus_Project") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

<h3>Imports<h3>

In [2]:
import sys
import gc
from pyspark.sql import DataFrame as SparkDF
sys.path.append('../work')
from config import db_properties,DB_CONFIG,jdbc_url
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pandas as pd
import numpy as np
import pyspark.pandas as ps
import requests
from pyspark.sql.types import FloatType
from pyspark.sql import Row
api_key = os.getenv("WARSAW_API_KEY")
import psycopg2
from psycopg2 import sql
import pyspark.sql.functions as sf
import zipfile
import urllib.request
from pyspark.testing import assertDataFrameEqual

<h2>Bus Live Data Table<h2>

<h3>Calling Warsaw API<h3>

In [3]:
url = "https://api.um.warszawa.pl/api/action/busestrams_get/"
params = {
    "resource_id": "f2e5503e-927d-4ad3-9500-4ab9e55ebe59", 
    "apikey": api_key,
    "type": "1" # '1' to autobusy, '2' to tramwaje
}
response = requests.get(url, params={"apikey": api_key,
            "resource_id":"f2e5503e927d-4ad3-9500-4ab9e55deb59","type": "1"},
            timeout=10) #Type 1 for BUS
if response.status_code == 200:
    print("API call successful")

API call successful


<h3>Creating df<h3>

In [4]:
if response.status_code == 200:
    data = response.json()
    if isinstance(data, dict) and "result" in data and isinstance(data["result"], list):
        print(data["result"])
        df_live = spark.createDataFrame(data["result"])

[{'Lines': '108', 'Lon': 21.073806, 'VehicleNumber': '1000', 'Time': '2026-04-08 20:16:52', 'Lat': 52.21204, 'Brigade': '1'}, {'Lines': '225', 'Lon': 21.115941, 'VehicleNumber': '1002', 'Time': '2026-04-08 18:12:01', 'Lat': 52.2344898, 'Brigade': '04'}, {'Lines': '219', 'Lon': 21.1449481, 'VehicleNumber': '1003', 'Time': '2026-04-08 20:16:53', 'Lat': 52.186471, 'Brigade': '3'}, {'Lines': '219', 'Lon': 21.10722, 'VehicleNumber': '1005', 'Time': '2026-04-08 20:16:53', 'Lat': 52.220371, 'Brigade': '1'}, {'Lines': '108', 'Lon': 21.023531, 'VehicleNumber': '1006', 'Time': '2026-04-08 20:16:52', 'Lat': 52.228153, 'Brigade': '2'}, {'Lines': '219', 'Lon': 21.102468, 'VehicleNumber': '1007', 'Time': '2026-04-08 20:16:42', 'Lat': 52.222735, 'Brigade': '2'}, {'Lines': 'L-8', 'Lon': 21.0143923, 'VehicleNumber': '10071', 'Time': '2026-04-08 20:16:55', 'Lat': 52.3877683, 'Brigade': '1'}, {'Lines': 'L31', 'Lon': 21.0405553, 'VehicleNumber': '10072', 'Time': '2026-04-08 20:16:31', 'Lat': 52.4332296, '

<h3>Writing to DB<h3>

In [5]:
df_live.write.jdbc(url=jdbc_url, table="bronze.live_bus_data", 
                   mode="overwrite", properties=db_properties)

<h3>Check<h3>

In [6]:
df_live_test = spark.read.jdbc(url=jdbc_url, table="bronze.live_bus_data", 
                               properties=db_properties)
print(assertDataFrameEqual(df_live_test, df_live))

None


In [7]:
spark.catalog.clearCache()
gc.collect()
spark.stop()